**Deliverable 3 - INF201**

Linda Qviller Halvorsen

Marita Nyjordet Pedersen


# Deliverable 03 – Object-Oriented Battle Simulator
### INF201 – Advanced Programming

## Introduction

In this assignment, we developed a mini-RPG battle simulator using object-oriented programming in Python.

The project demonstrates inheritance, encapsulation, object composition, and interaction between classes.

We implemented the required `Dice`, `Weapon`, `Character`, `Warrior`, `Mage`, and `Battle` classes, along with some additional functionality.

This notebook presents the classes, their attributes and methods, and tests their functionality. We also explain our design decisions and demonstrate battles with and without weapons.



## 1. Imports and Class Inspection

The classes are implemented in separate Python modules, with the item classes collected in `items.py`.

We import the classes and use a helper function to display the attributes, methods, and properties of an object.

This allows us to demonstrate the structure of each class without modifying its implementation.


In [23]:

import inspect

from dice import Dice
from items import Item, Weapon
from character import Character
from warrior import Warrior
from mage import Mage
from battle import Battle


def show_class(instance):
    """Display an object's attributes, methods, and properties."""

    cls = type(instance)

    methods = sorted(
        name
        for name, member in inspect.getmembers(
            cls, inspect.isfunction
        )
        if not name.startswith("_")
        or name in ("__repr__", "__call__", "__str__")
    )

    properties = sorted(
        name
        for name, member in inspect.getmembers(cls)
        if isinstance(member, property)
    )

    print(f"Class: {cls.__name__}")
    print("Attributes:", vars(instance))
    print("Methods:", methods)
    print("Properties:", properties)


print("All classes imported successfully.")


All classes imported successfully.



## 2. Dice Class

The `Dice` class represents a dice with a configurable number of sides.

The default is six sides, but any number of sides greater than or equal to two is allowed.

**Main functionality:**
- `roll()` generates a random integer between 1 and the number of sides.
- `__repr__()` returns a readable representation of the dice.
- `__call__()` allows the object to be called directly to roll the dice.

**Design decision:**

We implemented `Dice` as a separate class so that different weapons can use different dice without duplicating the random damage calculation.


In [24]:

print("=== DICE CLASS TEST ===")

dice = Dice()
d20 = Dice(20)

show_class(dice)

print("\nDefault dice:", dice)
print("D20 dice:", d20)

print("\nRoll using roll():", d20.roll())
print("Roll using __call__():", d20())

# Test valid dice rolls
results = [d20.roll() for _ in range(100)]

assert all(1 <= result <= 20 for result in results)
assert repr(dice) == "Dice(sides=6)"

# Test invalid number of sides
try:
    Dice(1)
except ValueError:
    print("\nInvalid number of sides correctly rejected.")
else:
    raise AssertionError("Dice(1) should raise ValueError")

print("\nAll Dice tests passed!")


=== DICE CLASS TEST ===
Class: Dice
Attributes: {'sides': 6}
Methods: ['__call__', '__repr__', 'roll']
Properties: []

Default dice: Dice(sides=6)
D20 dice: Dice(sides=20)

Roll using roll(): 1
Roll using __call__(): 12

Invalid number of sides correctly rejected.

All Dice tests passed!



## 3. Item Class

The `Item` class is a general superclass for items in the game.

It contains the shared `name` attribute and a `__repr__()` method.

**Design decision:**

Although the assignment only requires the `Weapon` class, we introduced `Item` as a common superclass.

This allows different item types to inherit shared functionality instead of implementing it separately.

We also organized the item classes in `items.py`, following the module structure suggested in the assignment.


In [25]:

print("=== ITEM CLASS TEST ===")

item = Item("Example Item")

show_class(item)

print("\nItem representation:", repr(item))

assert item.name == "Example Item"

print("\nAll Item tests passed!")


=== ITEM CLASS TEST ===
Class: Item
Attributes: {'name': 'Example Item'}
Methods: ['__repr__']
Properties: []

Item representation: Item(name='Example Item')

All Item tests passed!



## 4. Weapon Class

The `Weapon` class inherits from `Item` and uses a `Dice` object to calculate damage.

When a character has a weapon equipped, the weapon damage replaces the character's base damage.

We extended the required functionality with:

- `damage_bonus`: Additional damage added to the dice roll.
- `min_damage`: Minimum damage before armor reduction.
- `special_effect`: An optional special weapon ability.
- `special_chance`: The probability of activating the special ability.

**Design decisions:**

We wanted equipped weapons to be stronger than the default base attack of 10 damage. Therefore, the weapons receive additional damage bonuses.

We also introduced different minimum damage values so that stronger weapons cannot produce the same low damage as weaker weapons.

Rather than creating a separate class for each weapon, we use one `Weapon` class with configurable attributes.

This keeps the implementation relatively simple while allowing different weapon behaviors.


In [26]:

print("=== WEAPON CLASS TEST ===")

# Create the three weapons
dagger = Weapon(
    name="Dagger",
    damage_dice=Dice(4),
    damage_bonus=10,
    min_damage=11,
    special_effect="extra_attack",
    special_chance=0.30
)

sword = Weapon(
    name="Sword",
    damage_dice=Dice(8),
    damage_bonus=10,
    min_damage=15,
    special_effect="ignore_armor",
    special_chance=0.20
)

axe = Weapon(
    name="Axe",
    damage_dice=Dice(12),
    damage_bonus=10,
    min_damage=19,
    special_effect="miss",
    special_chance=0.25
)

show_class(sword)

print("\n=== WEAPON DAMAGE COMPARISON ===")

for weapon, lower, upper in (
    (dagger, 11, 14),
    (sword, 15, 18),
    (axe, 19, 22)
):
    results = [weapon.roll_damage() for _ in range(100)]

    print(f"\nWeapon: {weapon.name}")
    print(f"Minimum observed: {min(results)}")
    print(f"Maximum observed: {max(results)}")
    print(f"Average damage: {sum(results) / len(results):.2f}")
    print(f"Expected damage range: {lower}-{upper}")

    assert all(lower <= damage <= upper for damage in results)

print("\nAll Weapon damage tests passed!")


=== WEAPON CLASS TEST ===
Class: Weapon
Attributes: {'name': 'Sword', 'damage_dice': Dice(sides=8), 'damage_bonus': 10, 'min_damage': 15, 'special_effect': 'ignore_armor', 'special_chance': 0.2}
Methods: ['__repr__', 'activate_special', 'roll_damage']
Properties: []

=== WEAPON DAMAGE COMPARISON ===

Weapon: Dagger
Minimum observed: 11
Maximum observed: 14
Average damage: 12.43
Expected damage range: 11-14

Weapon: Sword
Minimum observed: 15
Maximum observed: 18
Average damage: 15.87
Expected damage range: 15-18

Weapon: Axe
Minimum observed: 19
Maximum observed: 22
Average damage: 19.55
Expected damage range: 19-22

All Weapon damage tests passed!



### 4.1 Weapon Special Abilities

We introduced three weapons with different abilities to provide strategic choices.

| Weapon | Damage | Special ability |
|---|---|---|
| Dagger | 11–14 | 30% chance of an additional attack |
| Sword | 15–18 | 20% chance to ignore armor |
| Axe | 19–22 | 25% chance to miss |

**Design decision:**

We wanted players to have reasons to choose different weapons instead of always selecting the weapon with the highest damage.

The Dagger can attack twice, the Sword can bypass armor, and the Axe offers higher damage at the risk of missing.

These abilities are optional extensions beyond the assignment requirements.

To test their functionality without relying on random activation, we temporarily set the activation probability to 100%.


In [27]:

print("=== WEAPON SPECIAL ABILITY TEST ===")

# Create weapons with guaranteed special effects
special_dagger = Weapon(
    "Dagger", Dice(4), 10, 11, "extra_attack", 1.0
)

special_sword = Weapon(
    "Sword", Dice(8), 10, 15, "ignore_armor", 1.0
)

special_axe = Weapon(
    "Axe", Dice(12), 10, 19, "miss", 1.0
)

for weapon in (special_dagger, special_sword, special_axe):

    attacker = Character("Attacker", weapon=weapon)
    defender = Warrior("Defender", armor=5)

    hp_before = defender.hp

    attacker.attack(defender)

    damage_dealt = hp_before - defender.hp

    print(f"\nWeapon: {weapon.name}")
    print(f"Damage dealt: {damage_dealt}")

    if weapon.name == "Dagger":
        assert 12 <= damage_dealt <= 18

    elif weapon.name == "Sword":
        assert 15 <= damage_dealt <= 18

    elif weapon.name == "Axe":
        assert damage_dealt == 0

print("\nAll weapon special ability tests passed!")


=== WEAPON SPECIAL ABILITY TEST ===
Attacker makes an extra attack with Dagger!

Weapon: Dagger
Damage dealt: 13
Attacker's Sword ignores armor!

Weapon: Sword
Damage dealt: 15
Attacker misses with Axe!

Weapon: Axe
Damage dealt: 0

All weapon special ability tests passed!



### 4.2 Additional Item Classes

The `items.py` module also contains three additional classes:

- **Armor:** Provides defense and damage bonuses.
- **ArcheryWeapon:** Inherits from `Weapon` and uses an additional dice roll to determine whether an attack hits.
- **Potion:** Provides healing or mana bonuses and can only be used once.

These classes were developed as possible extensions of the item system but are not used in the current game. We have therefore chosen not to include separate tests of these classes in this notebook.



## 5. Character Class

The `Character` class is the base class for characters in the game.

It contains attributes such as `name`, `_hp`, `hp_max`, `damage`, and `weapon`.

**Main functionality:**
- `take_damage()` reduces HP without allowing negative values.
- `heal()` restores HP without exceeding the maximum.
- `attack()` attacks another character using base damage or weapon damage.
- `hp` provides read-only access to the character's current health.
- `is_alive` returns `True` when HP is greater than zero.

**Design decisions:**

We use the protected attribute `_hp` to demonstrate encapsulation.

Instead of modifying another character's HP directly, the `attack()` method calls the target's `take_damage()` method.

We also introduced an optional `weapon` attribute. This allows characters to fight with or without weapons.

When a weapon is equipped, dice-based weapon damage replaces the character's base damage, as required by the assignment.


In [28]:

print("=== CHARACTER CLASS TEST ===")

attacker = Character("Attacker")
target = Character("Target")

show_class(attacker)

print("\nInitial target HP:", target.hp)

# Test attack without weapon
attacker.attack(target)

print("HP after attack:", target.hp)

assert target.hp == 90

# Test healing
target.heal(100)

print("\nHP after healing:", target.hp)

assert target.hp == 100

# Test damage greater than remaining HP
target.take_damage(200)

print("HP after fatal damage:", target.hp)
print("Is alive:", target.is_alive)

assert target.hp == 0
assert target.is_alive is False

# Test read-only properties
try:
    target.hp = 50
except AttributeError:
    print("\nHP property is read-only.")
else:
    raise AssertionError("HP should be read-only")

# Test weapon-based damage
armed_attacker = Character(
    name="Armed Attacker",
    weapon=sword
)

armed_target = Character("Armed Target")

hp_before = armed_target.hp
armed_attacker.attack(armed_target)

damage_dealt = hp_before - armed_target.hp

print("\nDamage with Sword:", damage_dealt)

assert 15 <= damage_dealt <= 18

print("\nAll Character tests passed!")


=== CHARACTER CLASS TEST ===
Class: Character
Attributes: {'name': 'Attacker', '_hp': 100, 'hp_max': 100, 'damage': 10, 'weapon': None}
Methods: ['__repr__', '__str__', 'attack', 'heal', 'introduce', 'take_damage']
Properties: ['hp', 'is_alive']

Initial target HP: 100
HP after attack: 90

HP after healing: 100
HP after fatal damage: 0
Is alive: False

HP property is read-only.
Armed Attacker's Sword ignores armor!

Damage with Sword: 15

All Character tests passed!



## 6. Warrior Class

The `Warrior` class inherits from `Character` and specializes in defense.

It introduces an `armor` attribute with a default value of 5.

The class overrides `take_damage()` to reduce incoming damage by the armor value before modifying HP.

**Design decisions:**

We use inheritance because a Warrior shares the general attributes and methods of `Character`.

By overriding `take_damage()`, we can introduce defensive behavior without rewriting the rest of the Character functionality.

We also added the optional `ignore_armor` parameter to support the Sword's special ability.

Under normal circumstances, the Warrior's armor reduces incoming damage.


In [29]:

print("=== WARRIOR CLASS TEST ===")

warrior = Warrior("Warrior")

show_class(warrior)

print("\nInitial HP:", warrior.hp)
print("Armor:", warrior.armor)

# Normal damage reduction
warrior.take_damage(12)

print("\nHP after 12 incoming damage:", warrior.hp)

assert warrior.hp == 93

# Attack ignoring armor
warrior.take_damage(12, ignore_armor=True)

print("HP after ignoring armor:", warrior.hp)

assert warrior.hp == 81

# Test damage smaller than armor
warrior.take_damage(3)

print("HP after 3 incoming damage:", warrior.hp)

assert warrior.hp == 81

print("\nAll Warrior tests passed!")


=== WARRIOR CLASS TEST ===
Class: Warrior
Attributes: {'name': 'Warrior', '_hp': 100, 'hp_max': 100, 'damage': 10, 'weapon': None, 'armor': 5}
Methods: ['__repr__', '__str__', 'attack', 'heal', 'introduce', 'take_damage']
Properties: ['hp', 'is_alive']

Initial HP: 100
Armor: 5

HP after 12 incoming damage: 93
HP after ignoring armor: 81
HP after 3 incoming damage: 81

All Warrior tests passed!



## 7. Mage Class

The `Mage` class inherits from `Character` and introduces mana-based attacks.

The class has two additional attributes:
- `mana`: Current mana.
- `mana_max`: Maximum mana.

Both have a default value of 30.

**Main functionality:**

A magical attack consumes 10 mana and deals double base damage without using the equipped weapon.

A regular attack uses the equipped weapon, if available, and regenerates 5 mana without exceeding `mana_max`.

If the Mage has less than 10 mana, a regular attack is used automatically.

**Design decisions:**

The assignment requires the Mage to use magic when enough mana is available.

We extended this functionality by allowing the player to choose between magical and regular attacks.

This introduces an additional strategic choice, since a player may prefer to use a weapon and regenerate mana.

For automatic battles, the default behavior still prioritizes magic when sufficient mana is available.


In [30]:

print("=== MAGE CLASS TEST ===")

mage = Mage(
    name="Mage",
    weapon=sword
)

training_target = Character(
    name="Training Target",
    hp_max=300
)

show_class(mage)

print("\nInitial mana:", mage.mana)

# Magic attack
hp_before = training_target.hp

mage.attack(training_target, use_magic=True)

print("\nMagic damage:", hp_before - training_target.hp)
print("Mana remaining:", mage.mana)

assert hp_before - training_target.hp == 20
assert mage.mana == 20

# Regular attack by choice
hp_before = training_target.hp

mage.attack(training_target, use_magic=False)

regular_damage = hp_before - training_target.hp

print("\nRegular attack damage:", regular_damage)
print("Mana after regular attack:", mage.mana)

assert 15 <= regular_damage <= 18
assert mage.mana == 25

# Use magic until mana is below 10
mage.attack(training_target, use_magic=True)
mage.attack(training_target, use_magic=True)

print("\nMana after two magic attacks:", mage.mana)

assert mage.mana == 5

# Automatic fallback to regular attack
hp_before = training_target.hp

mage.attack(training_target, use_magic=True)

fallback_damage = hp_before - training_target.hp

print("\nAutomatic regular attack damage:", fallback_damage)
print("Mana after automatic regular attack:", mage.mana)

assert 15 <= fallback_damage <= 18
assert mage.mana == 10

# Verify maximum mana
for _ in range(5):
    mage.attack(training_target, use_magic=False)

print("\nMana after repeated regular attacks:", mage.mana)

assert mage.mana == mage.mana_max == 30

print("\nAll Mage tests passed!")


=== MAGE CLASS TEST ===
Class: Mage
Attributes: {'name': 'Mage', '_hp': 100, 'hp_max': 100, 'damage': 10, 'weapon': Weapon(name='Sword', damage_dice=Dice(sides=8), damage_bonus=10, min_damage=15, special_effect='ignore_armor', special_chance=0.2), 'mana_max': 30, 'mana': 30}
Methods: ['__repr__', '__str__', 'attack', 'heal', 'introduce', 'take_damage']
Properties: ['hp', 'is_alive']

Initial mana: 30

Magic damage: 20
Mana remaining: 20
Mage's Sword ignores armor!

Regular attack damage: 15
Mana after regular attack: 25

Mana after two magic attacks: 5

Automatic regular attack damage: 15
Mana after automatic regular attack: 10

Mana after repeated regular attacks: 30

All Mage tests passed!




## 8. Battle Class

The `Battle` class manages turn-based combat between two characters.

It is initialized with a `hero` and an `enemy`, both of which are `Character` objects or subclasses.

**Main functionality:**
- `perform_attack()` executes one character's attack.
- `play_round()` lets Hero attack first, followed by Enemy if still alive.
- `simulate_battle()` runs the battle until one character is defeated or the battle is exited.
- `round_number` tracks the number of rounds.
- `exited` records whether a player has ended the battle early.

**Design decisions:**

We created a separate `Battle` class to manage combat independently of the characters and the user interface.

The class controls the order of attacks and ensures that a defeated character cannot attack.

We use a `while` loop in `simulate_battle()` to continue the battle until one character reaches zero HP.

We also allow optional player input through callback functions. This makes it possible to use the same battle manager for automatic and player-controlled battles.

Additionally, we implemented an option to exit an ongoing battle without declaring a winner.



In [31]:

print("=== BATTLE CLASS TEST ===")

# Test that Hero attacks first
hero = Character(
    name="Hero",
    damage=100
)

enemy = Character(
    name="Enemy",
    hp_max=20
)

battle = Battle(
    hero=hero,
    enemy=enemy
)

show_class(battle)

# Execute one round
battle.play_round()

print("\nRound number:", battle.round_number)
print("Hero HP:", hero.hp)
print("Enemy HP:", enemy.hp)

# Enemy should not attack after being defeated
assert hero.hp == 100
assert enemy.hp == 0
assert battle.round_number == 1

# Test early exit
exit_battle = Battle(
    Character("Hero"),
    Character("Enemy")
)

winner = exit_battle.simulate_battle(
    choose_hero=lambda character: None
)

assert exit_battle.exited is True
assert winner is None
assert exit_battle.hero.hp == 100
assert exit_battle.enemy.hp == 100

print("\nAll Battle tests passed!")


=== BATTLE CLASS TEST ===
Class: Battle
Attributes: {'hero': <Character name=Hero hp=100 damage=100>, 'enemy': <Character name=Enemy hp=20 damage=10>, 'round_number': 0, 'exited': False}
Methods: ['perform_attack', 'play_round', 'simulate_battle']
Properties: []

========== ROUND 1 ==========

========== HERO'S TURN ==========
Hero attacks automatically.
Damage dealt: 20
Enemy HP: 0/20

========== ROUND STATUS ==========
Hero: 100/100 HP
Enemy: 0/20 HP

Round number: 1
Hero HP: 100
Enemy HP: 0

========== ROUND 1 ==========

========== HERO'S TURN ==========

========== BATTLE EXITED ==========
The battle ended without a winner.

All Battle tests passed!



## 9. Battle Simulations

We demonstrate battles between different character types, both with and without weapons.

The simulations use the `Battle` class, which manages the attack order and continues the battle until one character reaches zero HP.

We create new characters for each battle so that previous tests do not affect the results.

### Simulation 1: Warrior vs Mage without weapons

This simulation demonstrates how the character specializations work using only their default attributes.

### Simulation 2: Mage vs Warrior with weapons

This simulation demonstrates object composition, dice-based damage, and weapon special abilities.

Because weapon damage and abilities involve randomness, the winner and number of rounds may vary between executions.


In [32]:

print("=== BATTLE SIMULATION 1 ===")
print("Warrior vs Mage without weapons")

warrior_no_weapon = Warrior("Warrior A")
mage_no_weapon = Mage("Mage A")

battle_no_weapons = Battle(
    hero=warrior_no_weapon,
    enemy=mage_no_weapon
)

winner_no_weapons = battle_no_weapons.simulate_battle()

print("\n=== FINAL RESULTS ===")
print("Winner:", winner_no_weapons.name)
print("Rounds:", battle_no_weapons.round_number)
print("Warrior HP:", warrior_no_weapon.hp)
print("Mage HP:", mage_no_weapon.hp)

assert winner_no_weapons.is_alive
assert not (
    warrior_no_weapon.is_alive
    and mage_no_weapon.is_alive
)

print("\nBattle without weapons completed!")


=== BATTLE SIMULATION 1 ===
Warrior vs Mage without weapons

========== ROUND 1 ==========

========== WARRIOR A'S TURN ==========
Warrior A attacks automatically.
Damage dealt: 10
Mage A HP: 90/100

========== MAGE A'S TURN ==========
Mage A attacks automatically.
Attack type: Magic
Mana remaining: 20/30
Damage dealt: 15
Warrior A HP: 85/100

========== ROUND STATUS ==========
Warrior A: 85/100 HP
Mage A: 90/100 HP

========== ROUND 2 ==========

========== WARRIOR A'S TURN ==========
Warrior A attacks automatically.
Damage dealt: 10
Mage A HP: 80/100

========== MAGE A'S TURN ==========
Mage A attacks automatically.
Attack type: Magic
Mana remaining: 10/30
Damage dealt: 15
Warrior A HP: 70/100

========== ROUND STATUS ==========
Warrior A: 70/100 HP
Mage A: 80/100 HP

========== ROUND 3 ==========

========== WARRIOR A'S TURN ==========
Warrior A attacks automatically.
Damage dealt: 10
Mage A HP: 70/100

========== MAGE A'S TURN ==========
Mage A attacks automatically.
Attack type: M

In [33]:

print("=== BATTLE SIMULATION 2 ===")
print("Mage vs Warrior with weapons")

mage_with_weapon = Mage(
    name="Mage B",
    weapon=dagger
)

warrior_with_weapon = Warrior(
    name="Warrior B",
    weapon=axe
)

battle_with_weapons = Battle(
    hero=mage_with_weapon,
    enemy=warrior_with_weapon
)

winner_with_weapons = battle_with_weapons.simulate_battle()

print("\n=== FINAL RESULTS ===")
print("Winner:", winner_with_weapons.name)
print("Rounds:", battle_with_weapons.round_number)
print("Mage HP:", mage_with_weapon.hp)
print("Warrior HP:", warrior_with_weapon.hp)

assert winner_with_weapons.is_alive
assert not (
    mage_with_weapon.is_alive
    and warrior_with_weapon.is_alive
)

print("\nBattle with weapons completed!")


=== BATTLE SIMULATION 2 ===
Mage vs Warrior with weapons

========== ROUND 1 ==========

========== MAGE B'S TURN ==========
Mage B attacks automatically.
Attack type: Magic
Mana remaining: 20/30
Damage dealt: 15
Warrior B HP: 85/100

========== WARRIOR B'S TURN ==========
Warrior B attacks automatically.
Warrior B misses with Axe!
Damage dealt: 0
Mage B HP: 100/100

========== ROUND STATUS ==========
Mage B: 100/100 HP
Warrior B: 85/100 HP

========== ROUND 2 ==========

========== MAGE B'S TURN ==========
Mage B attacks automatically.
Attack type: Magic
Mana remaining: 10/30
Damage dealt: 15
Warrior B HP: 70/100

========== WARRIOR B'S TURN ==========
Warrior B attacks automatically.
Damage dealt: 19
Mage B HP: 81/100

========== ROUND STATUS ==========
Mage B: 81/100 HP
Warrior B: 70/100 HP

========== ROUND 3 ==========

========== MAGE B'S TURN ==========
Mage B attacks automatically.
Attack type: Magic
Mana remaining: 0/30
Damage dealt: 15
Warrior B HP: 55/100

========== WARRIOR


## 10. Game Interface

The `game.py` module provides an interactive interface for the battle simulator.

It connects the character classes, weapons, and `Battle` class into a playable game.

**Main functionality:**
- Three game modes: Player vs Computer, Computer vs Computer, and Player vs Player.
- Selection of character class, name, and weapon.
- Selection between magic and regular attacks for Mage.
- Option to fight without weapons.
- Option to exit an ongoing battle.
- Option to start a new game after a battle.

**Design decisions:**

Although an interactive game interface was not required by the assignment, we chose to implement `game.py` to demonstrate how the classes work together in a complete application.

We wanted players to be able to choose their characters and weapons rather than only observe predefined battle simulations.

We also introduced three game modes to demonstrate how the same classes and battle mechanics can be used in different situations.

To keep the responsibilities separated, `game.py` handles the menus and player choices, while the `Battle` class manages the actual combat.

This allows us to reuse the same battle system across all three game modes.

The interactive game can be started from the terminal using:

`python game.py`



## 11. Game Balance and Limitations

The main objective of this assignment was to demonstrate object-oriented programming principles rather than to create a fully balanced RPG.

However, we wanted to investigate how character attributes, weapons, and attack strategies affected the outcome of battles.

### Balance Testing

We performed exploratory simulations involving thousands of battles with different character types and weapon combinations.

We investigated:
- Warrior versus Mage with and without weapons.
- Different combinations of Dagger, Sword, and Axe.
- The advantage of attacking first.
- Different attack strategies for Mage.
- Battles between characters of the same class.

### Observations

The results showed that attack order and weapon selection can significantly influence the outcome.

In battles without weapons, the character attacking first won every time under the tested default settings.

When both characters used weapons, Warrior generally performed better than Mage. The Warrior's armor and reliable weapon damage provided a significant advantage.

We also compared different Mage strategies. Using only regular attacks generally produced worse results than using magic whenever possible, while a mixed strategy gave similar results to prioritizing magic.

The weapon comparisons showed that Sword performed well for Warrior, while Dagger was competitive for Mage against other Mages.

Axe produced more balanced results in several combinations, partly because its higher damage is offset by a chance of missing.

### Limitations and Further Development

These results show that the game is not fully balanced.

A more advanced RPG would require further testing and adjustments to weapon damage, special abilities, character attributes, and attack strategies.

However, since game balance was not an explicit requirement of this assignment, we chose to retain the current mechanics.

Our priority was to demonstrate the required object-oriented programming concepts and ensure that the classes work correctly together.
